# Initialization and Optimizers: Where to Start, and How to Step

The runnable code of [Initialization and Optimizers: Where to Start, and How to Step](https://residual-stream.lvtan71.workers.dev/posts/initialization-and-optimizers/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/initialization-and-optimizers.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 3 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "initialization-and-optimizers"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/learning-is-optimization/a_loss.py', 'shared/initialization-and-optimizers/decay.py', 'shared/initialization-and-optimizers/deep.py', 'shared/initialization-and-optimizers/deep_init.py', 'shared/initialization-and-optimizers/optim.py', 'shared/initialization-and-optimizers/race.py', 'shared/softmax-and-cross-entropy/router4.py', 'shared/features-overfitting-regularization/router_data.py', 'shared/softmax-and-cross-entropy/softmax_basics.py', 'data/router-questions.csv']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Why depth breaks the signal

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/initialization-and-optimizers/#why-depth-breaks-the-signal)

In [ ]:
import numpy as np

def signal(scale, act, depth=20, width=256, seed=0):
    """Push a batch of 512 random inputs through `depth` layers. Return the size
    of the activations after each layer, and of the gradient arriving at each
    layer on the way back, starting from a random unit-size gradient at the top."""
    rng = np.random.default_rng(seed)
    x = rng.normal(size=(512, width))
    Ws = [rng.normal(0, scale(width), (width, width)) for _ in range(depth)]
    phi, dphi = {"relu": (lambda z: np.maximum(z, 0), lambda z: (z > 0) * 1.0),
                 "tanh": (np.tanh, lambda z: 1 - np.tanh(z) ** 2)}[act]
    rms = lambda a: np.sqrt((a ** 2).mean())
    h, zs, forward = x, [], []
    for W in Ws:
        zs.append(h @ W)
        h = phi(zs[-1])
        forward.append(rms(h))
    g, backward = rng.normal(size=h.shape), []
    for W, z in zip(Ws[::-1], zs[::-1]):
        g = (g * dphi(z)) @ W.T                        # 3.1's rules: mask, then W transposed
        backward.append(rms(g))
    return forward, backward[::-1]                      # backward[0] reaches layer 1

schemes = {
    "N(0, 0.01^2), ReLU": (lambda n: 0.01, "relu"),
    "N(0, 1), ReLU":      (lambda n: 1.0, "relu"),
    "Xavier 1/n, ReLU":   (lambda n: np.sqrt(1 / n), "relu"),
    "He 2/n, ReLU":       (lambda n: np.sqrt(2 / n), "relu"),
    "Xavier 1/n, tanh":   (lambda n: np.sqrt(1 / n), "tanh"),
}
print(f"{'':19s} {'activation size after layer':^39s}  {'gradient reaching':^19s}")
print(f"{'':19s} {'1':>9s} {'5':>9s} {'10':>9s} {'20':>9s}  {'layer 20':>9s} {'layer 1':>9s}")
for name, (scale, act) in schemes.items():
    runs = [signal(scale, act, seed=s) for s in range(5)]
    f = np.median([r[0] for r in runs], axis=0)       # median over 5 seeds
    b = np.median([r[1] for r in runs], axis=0)
    print(f"{name:19s} " + " ".join(f"{f[k]:9.2e}" for k in (0, 4, 9, 19)) +
          f"  {b[19]:9.2e} {b[0]:9.2e}")

The post printed:

```text
                          activation size after layer         gradient reaching 
                            1         5        10        20   layer 20   layer 1
N(0, 0.01^2), ReLU   1.13e-01  1.98e-05  3.68e-10  1.37e-19   1.14e-01  1.21e-19
N(0, 1), ReLU        1.13e+01  1.98e+05  3.68e+10  1.37e+21   1.14e+01  1.21e+21
Xavier 1/n, ReLU     7.07e-01  1.89e-01  3.35e-02  1.14e-03   7.15e-01  1.00e-03
He 2/n, ReLU         1.00e+00  1.07e+00  1.07e+00  1.16e+00   1.01e+00  1.03e+00
Xavier 1/n, tanh     6.28e-01  3.21e-01  2.27e-01  1.58e-01   9.73e-01  1.88e-01
```

In [ ]:
%%writefile deep_init.py
import numpy as np
from router_data import split
from router4 import four_way
from deep import loss_and_grads
from optim import SGD, Adam

def init(sizes, rng, std):
    return [{"W": rng.normal(0, std(m), (m, n)), "b": np.zeros(n)}
            for m, n in zip(sizes[:-1], sizes[1:])]

if __name__ == "__main__":
    inits = {"N(0, 0.01^2)": lambda m: 0.01, "Xavier 1/n": lambda m: np.sqrt(1 / m),
             "He 2/n": lambda m: np.sqrt(2 / m)}
    sizes = [None] + [64] * 10 + [4]                         # ten hidden ReLU layers
    out = {}
    for seed in range(5):
        ((Xtr, ytr), _, _), _ = four_way(*split(seed))
        sizes[0] = Xtr.shape[1]
        for name, std in inits.items():
            for opt_name, make in [("SGD 0.1", lambda: SGD(0.1)), ("Adam 0.001", lambda: Adam(1e-3))]:
                rng = np.random.default_rng(seed)
                layers, opt = init(sizes, rng, std), make()
                start, g0 = loss_and_grads(layers, Xtr, ytr)
                first = g0[0]["W"]
                early = None
                for t in range(1000):
                    if t == 100:
                        early = loss_and_grads(layers, Xtr, ytr)[0]
                    idx = rng.choice(len(ytr), 16, replace=False)
                    _, grads = loss_and_grads(layers, Xtr[idx], ytr[idx])
                    for i, (L, G) in enumerate(zip(layers, grads)):
                        for k in L:
                            opt.step(L[k], G[k], (i, k))
                out.setdefault((name, opt_name), []).append(
                    (np.abs(first).max(), start, early, loss_and_grads(layers, Xtr, ytr)[0]))
    print(f"{'':14s} {'optimizer':11s} {'largest first-layer gradient':>29s}   train loss at 0 / 100 / 1,000 steps")
    for (name, opt_name), rows in out.items():
        r = np.median(rows, axis=0)
        print(f"{name:14s} {opt_name:11s} {r[0]:29.1e}   {r[1]:12.3f} {r[2]:7.3f} {r[3]:7.3f}")

In [ ]:
%run deep_init.py

The post printed:

```text
               optimizer    largest first-layer gradient   train loss at 0 / 100 / 1,000 steps
N(0, 0.01^2)   SGD 0.1                           7.3e-15          1.386   1.387   1.387
N(0, 0.01^2)   Adam 0.001                        7.3e-15          1.386   1.386   1.386
Xavier 1/n     SGD 0.1                           6.9e-04          1.386   1.376   0.001
Xavier 1/n     Adam 0.001                        6.9e-04          1.386   0.091   0.000
He 2/n         SGD 0.1                           2.5e-02          1.399   0.700   0.000
He 2/n         Adam 0.001                        2.5e-02          1.399   0.011   0.000
```

## Momentum: remember the last step

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/initialization-and-optimizers/#momentum-remember-the-last-step)

In [ ]:
import numpy as np
from a_loss import words as x, tokens as y

# "Learning Is Optimization"'s raw-feature fit: tokens = w * words + b, MSE loss, kappa = 128.
X = np.c_[x, np.ones_like(x)]
H = 2 / len(x) * X.T @ X                              # the Hessian
lo, hi = np.linalg.eigvalsh(H)
theta_star = np.linalg.solve(X.T @ X, X.T @ y)
grad = lambda th: 2 / len(x) * X.T @ (X @ th - y)
print(f"eigenvalues {lo:.4f} and {hi:.2f}, kappa {hi / lo:.0f}")

def steps_to(update, tol=1e-6, limit=100_000):
    """Steps until the parameters are within tol (relative) of the least-squares fit."""
    th, state = np.zeros(2), {}
    for t in range(1, limit + 1):
        th = update(th, state, t)
        if np.linalg.norm(th - theta_star) < tol * np.linalg.norm(theta_star):
            return t
    return None

def gd(eta):
    return lambda th, s, t: th - eta * grad(th)

def heavy_ball(eta, beta):
    def step(th, s, t):
        s["v"] = beta * s.get("v", 0) + grad(th)     # velocity: a running sum of gradients
        return th - eta * s["v"]
    return step

k = hi / lo
eta_gd = 2 / (hi + lo)                                # the best fixed step for plain descent
eta_hb = 4 / (np.sqrt(hi) + np.sqrt(lo)) ** 2         # Polyak's best choice for momentum
beta_hb = ((np.sqrt(k) - 1) / (np.sqrt(k) + 1)) ** 2
print(f"gradient descent, eta {eta_gd:.4f}:            {steps_to(gd(eta_gd)):5d} steps   "
      f"(theory: error shrinks by {(k - 1) / (k + 1):.4f} per step)")
print(f"momentum, eta {eta_hb:.4f}, beta {beta_hb:.3f}:   {steps_to(heavy_ball(eta_hb, beta_hb)):5d} steps   "
      f"(theory: {(np.sqrt(k) - 1) / (np.sqrt(k) + 1):.4f} per step)")
for beta in (0.5, 0.9, 0.99):
    n = steps_to(heavy_ball(eta_gd * (1 - beta), beta))
    print(f"momentum, beta {beta}, same effective step:   {n if n else 'no convergence':>5} steps")
# Tuned this way, the extreme directions sit exactly on the edge of the band,
# where the two roots coincide and the error decays like t * sqrt(beta)^t.
r, t = np.sqrt(beta_hb), np.arange(1, 1000)
print(f"first t with sqrt(beta)^t < 1e-6: {t[r ** t < 1e-6][0]}; with t * sqrt(beta)^t < 1e-6: "
      f"{t[t * r ** t < 1e-6][0]}")

The post printed:

```text
eigenvalues 0.2407 and 30.72, kappa 128
gradient descent, eta 0.0646:              882 steps   (theory: error shrinks by 0.9844 per step)
momentum, eta 0.1099, beta 0.701:     108 steps   (theory: 0.8374 per step)
momentum, beta 0.5, same effective step:     736 steps
momentum, beta 0.9, same effective step:     630 steps
momentum, beta 0.99, same effective step:    2302 steps
first t with sqrt(beta)^t < 1e-6: 78; with t * sqrt(beta)^t < 1e-6: 104
```

## Adam: a step size for every weight

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/initialization-and-optimizers/#adam-a-step-size-for-every-weight)

In [ ]:
import numpy as np

# Adam's moving averages on a gradient that is always 0.001: a small, perfectly
# consistent gradient. Both averages start at 0, so both start too small, and
# v (averaged with 0.999) far more than m (0.9). Without correction, the first
# steps are too large.
b1, b2, eta, g = 0.9, 0.999, 0.01, 0.001
m = v = 0.0
for t in range(1, 6):
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g * g
    raw, fixed = m / np.sqrt(v), (m / (1 - b1 ** t)) / np.sqrt(v / (1 - b2 ** t))
    print(f"t={t}: m {m:.2e}  v {v:.2e}   step without correction {eta * raw:.5f}"
          f"   with correction {eta * fixed:.5f}")

# The first corrected step is eta * g / |g| for every parameter, whatever the gradient's size.
for g in (1e-6, 1e-2, 10.0):
    m, v = (1 - b1) * g, (1 - b2) * g * g
    print(f"gradient {g:>7g}: first Adam step {eta * (m / (1 - b1)) / np.sqrt(v / (1 - b2)):.4f}"
          f"   first gradient-descent step {eta * g:.1e}")

# How long does the uncorrected step stay too large? Its ratio to the
# corrected one is (1 - b1^t) / sqrt(1 - b2^t).
t = np.arange(1, 1001)
ratio = (1 - b1 ** t) / np.sqrt(1 - b2 ** t)
print(f"uncorrected / corrected: peak {ratio.max():.2f} at t={t[ratio.argmax()]}, "
      f"{ratio[99]:.2f} at t=100, {ratio[999]:.2f} at t=1000")

The post printed:

```text
t=1: m 1.00e-04  v 1.00e-09   step without correction 0.03162   with correction 0.01000
t=2: m 1.90e-04  v 2.00e-09   step without correction 0.04250   with correction 0.01000
t=3: m 2.71e-04  v 3.00e-09   step without correction 0.04950   with correction 0.01000
t=4: m 3.44e-04  v 3.99e-09   step without correction 0.05442   with correction 0.01000
t=5: m 4.10e-04  v 4.99e-09   step without correction 0.05797   with correction 0.01000
gradient   1e-06: first Adam step 0.0100   first gradient-descent step 1.0e-08
gradient    0.01: first Adam step 0.0100   first gradient-descent step 1.0e-04
gradient      10: first Adam step 0.0100   first gradient-descent step 1.0e-01
uncorrected / corrected: peak 6.57 at t=12, 3.24 at t=100, 1.26 at t=1000
```

In [ ]:
import numpy as np
from router_data import split
from router4 import four_way
from deep import loss_and_grads
from deep_init import init
from optim import Adam

# Is Adam's epsilon what keeps it stuck at the tiny start? Remove it and see.
sizes = [None] + [64] * 10 + [4]
for eps in (1e-8, 1e-30):
    finals = []
    for seed in range(5):
        ((X, y), _, _), _ = four_way(*split(seed))
        sizes[0] = X.shape[1]
        rng = np.random.default_rng(seed)
        layers, opt = init(sizes, rng, lambda m: 0.01), Adam(1e-3, eps=eps)
        for t in range(1000):
            idx = rng.choice(len(y), 16, replace=False)
            _, grads = loss_and_grads(layers, X[idx], y[idx])
            for i, (L, G) in enumerate(zip(layers, grads)):
                for k in L:
                    opt.step(L[k], G[k], (i, k))
        finals.append(loss_and_grads(layers, X, y)[0])
    print(f"Adam eps {eps:g}: train loss after 1,000 steps, 5 seeds: " + " ".join(f"{v:.3f}" for v in finals))

The post printed:

```text
Adam eps 1e-08: train loss after 1,000 steps, 5 seeds: 1.386 1.386 1.386 1.386 1.386
Adam eps 1e-30: train loss after 1,000 steps, 5 seeds: 0.169 1.386 1.386 0.067 1.386
```

## When the optimizer matters

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/initialization-and-optimizers/#when-the-optimizer-matters)

In [ ]:
%%writefile optim.py
# optim.py
import numpy as np

class SGD:
    def __init__(self, eta):
        self.eta = eta
    def step(self, p, g, key):
        p -= self.eta * g

class Momentum:
    def __init__(self, eta, beta=0.9):
        self.eta, self.beta, self.v = eta, beta, {}
    def step(self, p, g, key):
        v = self.v[key] = self.beta * self.v.get(key, 0) + g
        p -= self.eta * v

def decays(key):
    """Weight decay applies to weight matrices only, not to biases or norm gains."""
    name = key[-1] if isinstance(key, tuple) else key
    return name == "W"

class Adam:
    def __init__(self, eta, beta1=0.9, beta2=0.999, eps=1e-8, decay=0.0, decoupled=True):
        self.eta, self.b1, self.b2, self.eps = eta, beta1, beta2, eps
        self.decay, self.decoupled, self.m, self.v, self.t = decay, decoupled, {}, {}, {}
    def step(self, p, g, key):
        if self.decay and not self.decoupled and decays(key):
            g = g + self.decay * p                              # L2 folded into the gradient
        t = self.t[key] = self.t.get(key, 0) + 1
        m = self.m[key] = self.b1 * self.m.get(key, 0) + (1 - self.b1) * g
        v = self.v[key] = self.b2 * self.v.get(key, 0) + (1 - self.b2) * g * g
        m_hat, v_hat = m / (1 - self.b1 ** t), v / (1 - self.b2 ** t)   # bias correction
        if self.decay and self.decoupled and decays(key):
            p -= self.eta * self.decay * p                      # AdamW: decay applied directly
        p -= self.eta * m_hat / (np.sqrt(v_hat) + self.eps)

In [ ]:
%%writefile deep.py
# deep.py
import numpy as np
from softmax_basics import softmax, cross_entropy

def init(sizes, rng):
    """He initialization for every ReLU layer: std sqrt(2 / fan_in)."""
    return [{"W": rng.normal(0, np.sqrt(2 / m), (m, n)), "b": np.zeros(n)}
            for m, n in zip(sizes[:-1], sizes[1:])]

def loss_and_grads(layers, X, y):
    """Softmax cross-entropy of a ReLU network, and its gradient ("Backpropagation, by Hand"'s rules)."""
    hs, h = [X], X
    for i, L in enumerate(layers):
        z = h @ L["W"] + L["b"]
        h = z if i == len(layers) - 1 else np.maximum(z, 0)
        hs.append(h)
    g = (softmax(hs[-1]) - np.eye(hs[-1].shape[1])[y]) / len(y)      # p - y, "Cross-Entropy and the Softmax Gradient"
    grads = [None] * len(layers)
    for i in reversed(range(len(layers))):
        grads[i] = {"W": hs[i].T @ g, "b": g.sum(0)}
        g = (g @ layers[i]["W"].T) * (hs[i] > 0)                         # back through ReLU
    return cross_entropy(hs[-1], y), grads

In [ ]:
%%writefile race.py
# race.py
import sys
import numpy as np
from router_data import split
from router4 import four_way
from softmax_basics import softmax, cross_entropy
from optim import SGD, Momentum, Adam

# The four-way router of "Cross-Entropy and the Softmax Gradient" with an L2 penalty: a convex objective, so
# "better optimizer" can only mean "lower objective sooner".
LAM = 0.003
def objective(W, b, X, y):
    return cross_entropy(X @ W + b, y) + 0.5 * LAM * (W ** 2).sum()

def run(opt, X, y, checkpoints=(30, 300, 3000)):
    W, b, out = np.zeros((X.shape[1], 4)), np.zeros(4), []
    for t in range(1, max(checkpoints) + 1):
        G = (softmax(X @ W + b) - np.eye(4)[y]) / len(y)
        opt.step(W, X.T @ G + LAM * W, "W")
        opt.step(b, G.sum(0), "b")
        if t in checkpoints:
            out.append(objective(W, b, X, y))
    return out

if __name__ == "__main__":
    with_length = len(sys.argv) > 1                       # add one raw feature: question length in characters
    print("plus raw question length:" if with_length else "bag of words:")
    etas = {"gradient descent": (SGD, [10 ** k for k in np.arange(-4, 1.6, 0.5)]),
            "momentum 0.9": (Momentum, [10 ** k for k in np.arange(-5, 1.1, 0.5)]),
            "Adam": (Adam, [10 ** k for k in np.arange(-3, 0.6, 0.5)])}
    for name, (make, grid) in etas.items():
        best = {}
        for eta in grid:
            rows = []
            for seed in range(5):
                tr, va, te = split(seed)
                ((X, y), _, _), _ = four_way(tr, va, te)
                if with_length:
                    X = np.c_[X, [len(r["question"]) for r in tr]]
                with np.errstate(over="ignore", invalid="ignore"):
                    rows.append(run(make(eta), X, y))
            r = np.nan_to_num(np.median(rows, axis=0), nan=np.inf)
            for k, v in enumerate(r):
                if v < best.get(k, (np.inf,))[0]:
                    best[k] = (v, eta)
        print(f"{name:17s} " + "   ".join(f"after {n:>4} steps {best[k][0]:.4f} (eta {best[k][1]:.2g})"
                                           for k, n in enumerate((30, 300, 3000))))

In [ ]:
%run race.py
%run race.py length

The post printed:

```text
bag of words:
gradient descent  after   30 steps 0.3052 (eta 10)   after  300 steps 0.3011 (eta 10)   after 3000 steps 0.3011 (eta 3.2)
momentum 0.9      after   30 steps 0.3105 (eta 3.2)   after  300 steps 0.3011 (eta 10)   after 3000 steps 0.3011 (eta 0.32)
Adam              after   30 steps 0.3158 (eta 0.1)   after  300 steps 0.3011 (eta 0.1)   after 3000 steps 0.3011 (eta 0.01)
plus raw question length:
gradient descent  after   30 steps 1.3734 (eta 0.0032)   after  300 steps 1.3021 (eta 0.0032)   after 3000 steps 0.8829 (eta 0.0032)
momentum 0.9      after   30 steps 1.3237 (eta 0.0032)   after  300 steps 0.8877 (eta 0.0032)   after 3000 steps 0.2948 (eta 0.01)
Adam              after   30 steps 0.3495 (eta 0.1)   after  300 steps 0.2917 (eta 0.32)   after 3000 steps 0.2914 (eta 0.01)
```

In [ ]:
import numpy as np
from router_data import split
from router4 import four_way

# The largest curvature of race.py's objective at the start (W = 0), where the
# softmax Hessian block is (diag(p) - p p^T) with p uniform, largest eigenvalue 1/4.
tr, va, te = split(0)
((X, y), _, _), _ = four_way(tr, va, te)
for name, Z in [("bag of words", X), ("plus raw length", np.c_[X, [len(r["question"]) for r in tr]])]:
    Zb = np.c_[Z, np.ones(len(Z))]
    top = 0.25 * np.linalg.eigvalsh(Zb.T @ Zb / len(Zb))[-1] + 0.003
    print(f"{name:16s} largest curvature {top:8.2f}, so gradient descent is stable only below 2 / that = {2 / top:.4f}")

The post printed:

```text
bag of words     largest curvature     0.51, so gradient descent is stable only below 2 / that = 3.9118
plus raw length  largest curvature   543.91, so gradient descent is stable only below 2 / that = 0.0037
```

In [ ]:
import numpy as np
from router_data import split
from router4 import four_way
from optim import SGD, Momentum, Adam
from race import run

# The curves behind the chart: bag of words plus raw question length, each
# optimizer at its best learning rate for 3,000 steps (from race.py), plus
# momentum at gradient descent's learning rate.
marks = [int(round(10 ** k)) for k in np.arange(0, 3.51, 0.25)]
for name, make in [("gradient descent", lambda: SGD(10 ** -2.5)),
                   ("momentum 0.9, same eta", lambda: Momentum(10 ** -2.5)),
                   ("momentum 0.9, eta 0.01", lambda: Momentum(0.01)), ("Adam", lambda: Adam(0.01))]:
    rows = []
    for seed in range(5):
        tr, va, te = split(seed)
        ((X, y), _, _), _ = four_way(tr, va, te)
        X = np.c_[X, [len(r["question"]) for r in tr]]
        rows.append(run(make(), X, y, checkpoints=marks))
    print(f"{name}: " + " ".join(f"{v:.4f}" for v in np.median(rows, axis=0)))
print("steps: " + " ".join(map(str, marks)))

The post printed:

```text
gradient descent: 1.3833 1.3821 1.3815 1.3804 1.3792 1.3769 1.3729 1.3661 1.3541 1.3333 1.2981 1.2400 1.1496 1.0234 0.8678
momentum 0.9, same eta: 1.3833 1.3863 1.3809 1.3796 1.3707 1.3541 1.3187 1.2596 1.1643 1.0320 0.8725 0.7008 0.5382 0.4079 0.3314
momentum 0.9, eta 0.01: 1.4433 1.7060 1.8667 6.4745 13.3828 12.4024 21.6123 14.2705 13.1169 6.2789 6.9338 0.4115 0.3242 0.3016 0.2945
Adam: 1.3531 1.2584 1.1951 1.0186 0.8411 0.5947 0.4377 0.3681 0.3287 0.3085 0.2982 0.2935 0.2917 0.2914 0.2914
steps: 1 2 3 6 10 18 32 56 100 178 316 562 1000 1778 3162
```

## Why training starts with a warmup

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/initialization-and-optimizers/#why-training-starts-with-a-warmup)

In [ ]:
import numpy as np

# 100,000 parameters whose gradients are pure noise of size 1: no direction is
# right, so ideally they shouldn't move much. Two things about Adam's early steps:
#   the typical step, m_hat / sqrt(v_hat), in units of eta;
#   the per-parameter rate 1 / sqrt(v_hat), whose true value here is 1. With few
#   samples, v_hat can be tiny by chance, and then that parameter's rate is huge.
rng = np.random.default_rng(0)
b1, b2, n = 0.9, 0.999, 100_000
m, v = np.zeros(n), np.zeros(n)
for t in range(1, 5001):
    g = rng.normal(size=n)
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g * g
    m_hat, v_hat = m / (1 - b1 ** t), v / (1 - b2 ** t)
    if t in (1, 2, 5, 10, 100, 1000, 5000):
        step = m_hat / np.sqrt(v_hat)
        print(f"step {t:5d}: typical update {np.sqrt((step ** 2).mean()):.3f} x eta;  "
              f"parameters with a rate over 3x its true value: {(1 / np.sqrt(v_hat) > 3).mean():6.2%}")
print(f"long-run update size, sqrt((1 - b1) / (1 + b1)) = {np.sqrt((1 - b1) / (1 + b1)):.3f}")

The post printed:

```text
step     1: typical update 1.000 x eta;  parameters with a rate over 3x its true value: 25.97%
step     2: typical update 0.708 x eta;  parameters with a rate over 3x its true value: 10.42%
step     5: typical update 0.452 x eta;  parameters with a rate over 3x its true value:  0.97%
step    10: typical update 0.329 x eta;  parameters with a rate over 3x its true value:  0.02%
step   100: typical update 0.231 x eta;  parameters with a rate over 3x its true value:  0.00%
step  1000: typical update 0.229 x eta;  parameters with a rate over 3x its true value:  0.00%
step  5000: typical update 0.229 x eta;  parameters with a rate over 3x its true value:  0.00%
long-run update size, sqrt((1 - b1) / (1 + b1)) = 0.229
```

## Weight decay, decoupled: AdamW

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/initialization-and-optimizers/#weight-decay-decoupled-adamw)

In [ ]:
%%writefile decay.py
# decay.py
import sys
import numpy as np
from router_data import split
from router4 import four_way
from softmax_basics import softmax, cross_entropy
from optim import Adam, SGD

STEPS = int(sys.argv[1]) if len(sys.argv) > 1 else 3000   # steps per run

class L2SGD(SGD):
    """Features and Overfitting's L2 penalty with plain mini-batch gradient descent."""
    def __init__(self, eta, lam):
        super().__init__(eta)
        self.lam = lam
    def step(self, p, g, key):
        super().step(p, g + (self.lam * p if key == "W" else 0), key)

def train(opt, X, y, seed, steps=STEPS, batch=16):
    rng = np.random.default_rng(seed)
    W, b = np.zeros((X.shape[1], 4)), np.zeros(4)
    for _ in range(steps):
        i = rng.choice(len(y), batch, replace=False)
        G = (softmax(X[i] @ W + b) - np.eye(4)[y[i]]) / batch
        opt.step(W, X[i].T @ G, "W")                    # the optimizer adds any decay itself
        opt.step(b, G.sum(0), "b")
    return W, b

METHODS = [("no decay", lambda lam: Adam(0.01), [0]),
           ("SGD + L2", lambda lam: L2SGD(0.5, lam), [0.0003, 0.001, 0.003]),
           ("Adam + L2", lambda lam: Adam(0.01, decay=lam, decoupled=False), [0.0001, 0.0003, 0.001, 0.003]),
           ("AdamW", lambda lam: Adam(0.01, decay=lam), [0.1, 0.3, 1])]
res = {}
for seed in range(20):
    ((Xtr, ytr), (Xva, yva), (Xte, yte)), _ = four_way(*split(seed))
    common = Xtr.sum(0) >= 5                                # words in 5+ training questions
    for name, make, lams in METHODS:
        for lam in lams:
            W, b = train(make(lam), Xtr, ytr, seed)
            size = lambda mask: np.sqrt((W[mask] ** 2).mean())
            res.setdefault((name, lam), []).append(
                (cross_entropy(Xva @ W + b, yva), size(common), size(~common),
                 cross_entropy(Xte @ W + b, yte), ((Xte @ W + b).argmax(1) == yte).mean()))
print(f"{STEPS:,} steps, mini-batches of 16")
for (name, lam), rows in res.items():
    r = np.array(rows)
    print(f"{name:9s} λ {lam:<6} val CE {r[:, 0].mean():.3f}   weight size: common words "
          f"{r[:, 1].mean():.3f}, rare words {r[:, 2].mean():.3f}, ratio {np.mean(r[:, 1] / r[:, 2]):.2f}")

corrected = np.sqrt(1 / 20 + 40 / 120)
se = lambda d: f"{d.mean():+.3f} ± {d.std(ddof=1) * corrected:.3f}"
best = {n: min((k for k in res if k[0] == n), key=lambda k: np.mean(np.array(res[k])[:, 0]))
        for n, _, _ in METHODS}
test = {n: np.array(res[k]) for n, k in best.items()}
print("chosen on validation: " + ", ".join(f"{n} λ {k[1]}" for n, k in best.items() if n != "no decay"))
print("on test, at the chosen λ:  " + ", ".join(f"{n} CE {t[:, 3].mean():.3f} acc {t[:, 4].mean():.3f}"
                                                for n, t in test.items()))
for a, b in [("AdamW", "SGD + L2"), ("AdamW", "no decay"), ("SGD + L2", "no decay")]:
    print(f"test, {a} minus {b}: CE {se(test[a][:, 3] - test[b][:, 3])}, "
          f"accuracy {se(test[a][:, 4] - test[b][:, 4])}")

In [ ]:
%run decay.py

The post printed:

```text
3,000 steps, mini-batches of 16
no decay  λ 0      val CE 0.881   weight size: common words 0.852, rare words 0.927, ratio 0.92
SGD + L2  λ 0.0003 val CE 0.914   weight size: common words 0.769, rare words 0.410, ratio 1.88
SGD + L2  λ 0.001  val CE 0.888   weight size: common words 0.626, rare words 0.328, ratio 1.91
SGD + L2  λ 0.003  val CE 0.893   weight size: common words 0.496, rare words 0.245, ratio 2.02
Adam + L2 λ 0.0001 val CE 0.912   weight size: common words 0.818, rare words 0.523, ratio 1.57
Adam + L2 λ 0.0003 val CE 0.907   weight size: common words 0.747, rare words 0.428, ratio 1.75
Adam + L2 λ 0.001  val CE 0.889   weight size: common words 0.624, rare words 0.333, ratio 1.88
Adam + L2 λ 0.003  val CE 0.894   weight size: common words 0.497, rare words 0.249, ratio 2.00
AdamW     λ 0.1    val CE 0.822   weight size: common words 0.613, rare words 0.664, ratio 0.92
AdamW     λ 0.3    val CE 0.819   weight size: common words 0.502, rare words 0.525, ratio 0.96
AdamW     λ 1      val CE 0.894   weight size: common words 0.307, rare words 0.312, ratio 0.98
chosen on validation: SGD + L2 λ 0.001, Adam + L2 λ 0.001, AdamW λ 0.3
on test, at the chosen λ:  no decay CE 0.870 acc 0.691, SGD + L2 CE 0.896 acc 0.640, Adam + L2 CE 0.897 acc 0.631, AdamW CE 0.822 acc 0.688
test, AdamW minus SGD + L2: CE -0.074 ± 0.042, accuracy +0.047 ± 0.040
test, AdamW minus no decay: CE -0.048 ± 0.047, accuracy -0.004 ± 0.017
test, SGD + L2 minus no decay: CE +0.026 ± 0.041, accuracy -0.051 ± 0.034
```

In [ ]:
%run decay.py 12000

The post printed:

```text
12,000 steps, mini-batches of 16
no decay  λ 0      val CE 1.128   weight size: common words 1.342, rare words 1.511, ratio 0.89
SGD + L2  λ 0.0003 val CE 0.912   weight size: common words 0.768, rare words 0.423, ratio 1.82
SGD + L2  λ 0.001  val CE 0.887   weight size: common words 0.624, rare words 0.329, ratio 1.90
SGD + L2  λ 0.003  val CE 0.892   weight size: common words 0.496, rare words 0.246, ratio 2.02
Adam + L2 λ 0.0001 val CE 0.958   weight size: common words 0.904, rare words 0.516, ratio 1.76
Adam + L2 λ 0.0003 val CE 0.916   weight size: common words 0.771, rare words 0.428, ratio 1.81
Adam + L2 λ 0.001  val CE 0.890   weight size: common words 0.628, rare words 0.333, ratio 1.89
Adam + L2 λ 0.003  val CE 0.894   weight size: common words 0.498, rare words 0.249, ratio 2.01
AdamW     λ 0.1    val CE 0.966   weight size: common words 1.025, rare words 1.093, ratio 0.94
AdamW     λ 0.3    val CE 0.855   weight size: common words 0.764, rare words 0.818, ratio 0.93
AdamW     λ 1      val CE 0.867   weight size: common words 0.340, rare words 0.361, ratio 0.94
chosen on validation: SGD + L2 λ 0.001, Adam + L2 λ 0.001, AdamW λ 0.3
on test, at the chosen λ:  no decay CE 1.098 acc 0.699, SGD + L2 CE 0.894 acc 0.634, Adam + L2 CE 0.892 acc 0.630, AdamW CE 0.856 acc 0.676
test, AdamW minus SGD + L2: CE -0.037 ± 0.060, accuracy +0.042 ± 0.040
test, AdamW minus no decay: CE -0.242 ± 0.098, accuracy -0.022 ± 0.029
test, SGD + L2 minus no decay: CE -0.205 ± 0.095, accuracy -0.065 ± 0.035
```

## What Adam costs

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/initialization-and-optimizers/#what-adam-costs)

In [ ]:
# Bytes of training state per parameter, mixed-precision training (ZeRO's accounting).
params = 8e9                                   # an 8B model
state = {"BF16 weights": 2, "BF16 gradients": 2,
         "FP32 master weights": 4, "Adam m (FP32)": 4, "Adam v (FP32)": 4}
for name, b in state.items():
    print(f"{name:20s} {b:2d} bytes/param  {b * params / 1e9:5.0f} GB")
total = sum(state.values())
print(f"{'total':20s} {total:2d} bytes/param  {total * params / 1e9:5.0f} GB, "
      f"{total * params / 80e9:.1f} H100s of memory before a single activation")

The post printed:

```text
BF16 weights          2 bytes/param     16 GB
BF16 gradients        2 bytes/param     16 GB
FP32 master weights   4 bytes/param     32 GB
Adam m (FP32)         4 bytes/param     32 GB
Adam v (FP32)         4 bytes/param     32 GB
total                16 bytes/param    128 GB, 1.6 H100s of memory before a single activation
```